<a href="https://colab.research.google.com/github/volsarino/-/blob/main/%E3%83%8F%E3%83%B3%E3%83%89%E3%83%88%E3%83%A9%E3%83%83%E3%82%AD%E3%83%B3%E3%82%B0%E3%83%A2%E3%83%87%E3%83%AB(%E4%BA%8B%E5%89%8D%E5%AD%A6%E7%BF%92%E6%B8%88%E3%81%BF%E3%83%A2%E3%83%87%E3%83%AB%E4%BD%BF%E7%94%A8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import cv2
import gc
import os
import timm
import numpy as np
from PIL import Image
from torchvision.transforms import v2
import glob
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
pretrained_models = timm.list_models(pretrained=True)
vit_models = timm.list_models('*vit*')
print(vit_models[:5])

['convit_base', 'convit_small', 'convit_tiny', 'crossvit_9_240', 'crossvit_9_dagger_240']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_dir='/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/train'
test_dir='/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/test'
valid_dir='/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/valid'

In [ ]:
class HandDataset(Dataset):
    def __init__(self, dir, img_size):
        self.xml_paths = sorted(glob.glob(os.path.join(dir, '*.xml')))
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.xml_paths)

    def __getitem__(self, idx):
        xml_path = self.xml_paths[idx]

        #対応する画像ファイルの検出
        img_path = xml_path.replace('.xml', '.jpg')
        if not os.path.exists(img_path):
            img_path = xml_path.replace('.xml', '.png')
        if not os.path.exists(img_path):
            img_path = xml_path.replace('.xml', '.jpeg')

        #PILでRGB形式として読み込み
        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size

        #前処理
        image_tensor = self.transform(image)

        #XML解析
        tree = ET.parse(xml_path)
        root = tree.getroot()
        boxes = []

        for obj in root.findall('object'):
            bndbox = obj.find('bndbox')
            xmin = float(bndbox.find('xmin').text)/orig_w
            ymin = float(bndbox.find('ymin').text)/orig_h
            xmax = float(bndbox.find('xmax').text)/orig_w
            ymax = float(bndbox.find('ymax').text)/orig_h

            cx=(xmin+xmax)/2.0
            cy=(ymin+ymax)/2.0
            bw=xmax-xmin
            bh=ymax-ymin
            boxes.append([
                max(0.0, min(1.0,cx)),
                max(0.0, min(1.0,cy)),
                max(0.0, min(1.0,bw)),
                max(0.0, min(1.0,bh))
            ])

        if len(boxes) > 0:
            bbox_tensor = torch.tensor(boxes[0], dtype=torch.float32)
            label_tensor = torch.tensor(1, dtype=torch.long)
        else:
            bbox_tensor = torch.tensor([0.0, 0.0, 0.0, 0.0], dtype=torch.float32)
            label_tensor = torch.tensor(0, dtype=torch.long)

        return image_tensor, label_tensor, bbox_tensor

In [ ]:
train_dataset=HandDataset(train_dir,224)
valid_dataset=HandDataset(valid_dir,224)

In [ ]:
print("train_dataset の枚数:", len(train_dataset))
print("valid_dataset の枚数:", len(valid_dataset))

train_dataset の枚数: 3840
valid_dataset の枚数: 480


In [ ]:
print("train_dir のパス:", train_dir)
print("train 内のファイル数:", len(os.listdir(train_dir)) if os.path.exists(train_dir) else "存在しません")

print("valid_dir のパス:", valid_dir)
print("valid 内のファイル数:", len(os.listdir(valid_dir)) if os.path.exists(valid_dir) else "存在しません")

train_dir のパス: /content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/train
train 内のファイル数: 7680
valid_dir のパス: /content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/valid
valid 内のファイル数: 960


In [ ]:
class ViTModel(nn.Module):
  def __init__(self,model_name='convit_base',pretrained=True):
    super(ViTModel,self).__init__()
    self.convit=timm.create_model(model_name,pretrained=pretrained,num_classes=0)
    in_features=self.convit.num_features
    self.classifier=nn.Sequential(
        nn.Linear(in_features,128),
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(128,2)
    )

    self.bbox_regressor=nn.Sequential(
        nn.Linear(in_features,128),
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(128,4),
        nn.Sigmoid()
    )

  def forward(self,x):
    features=self.convit(x)
    class_logits=self.classifier(features)
    bbox_pred=self.bbox_regressor(features)
    return class_logits,bbox_pred




In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=ViTModel(model_name='convit_base',pretrained=True).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
criterion_cls=nn.CrossEntropyLoss()
criterion_bbox=nn.SmoothL1Loss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-5,weight_decay=1e-2)

In [ ]:
batch_size = 16
accumulation_steps = 2
epochs = 10

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

#損失関数・オプティマイザ・スケジューラの定義
criterion_cls = nn.CrossEntropyLoss()
criterion_bbox = nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler('cuda')

torch.cuda.empty_cache()
gc.collect()

for epoch in range(epochs):
    #学習フェーズ
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for i, (images, labels, bbox) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        bbox = bbox.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            class_logits, bbox_pred = model(images)
            loss_cls = criterion_cls(class_logits, labels)
            loss_bbox = criterion_bbox(bbox_pred, bbox)
            total_loss = loss_cls + 5.0 * loss_bbox

        scaler.scale(total_loss).backward()

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += total_loss.item() * accumulation_steps

    avg_train_loss = train_loss / len(train_loader)

    # エポック終了時に1回のみ更新
    scheduler.step()

    #検証フェーズ
    model.eval()
    valid_loss = 0.0

    with torch.no_grad():
        for images, labels, bbox in valid_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            bbox = bbox.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                class_logits, bbox_pred = model(images)
                loss_cls = criterion_cls(class_logits, labels)
                loss_bbox = criterion_bbox(bbox_pred, bbox)
                total_loss = loss_cls + 2.0 * loss_bbox

            valid_loss += total_loss.item()

    avg_valid_loss = valid_loss / len(valid_loader)
    torch.cuda.empty_cache()
    gc.collect()

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch+1:02d}/{epochs:02d}] | Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f} | LR: {current_lr:.6f}')

Epoch [01/10] | Train Loss: 0.0439 | Valid Loss: 0.0198 | LR: 0.000098
Epoch [02/10] | Train Loss: 0.0465 | Valid Loss: 0.0195 | LR: 0.000090
Epoch [03/10] | Train Loss: 0.0403 | Valid Loss: 0.0193 | LR: 0.000079
Epoch [04/10] | Train Loss: 0.0288 | Valid Loss: 0.0147 | LR: 0.000065
Epoch [05/10] | Train Loss: 0.0180 | Valid Loss: 0.0125 | LR: 0.000050
Epoch [06/10] | Train Loss: 0.0124 | Valid Loss: 0.0116 | LR: 0.000035
Epoch [07/10] | Train Loss: 0.0094 | Valid Loss: 0.0117 | LR: 0.000021
Epoch [08/10] | Train Loss: 0.0082 | Valid Loss: 0.0119 | LR: 0.000010
Epoch [09/10] | Train Loss: 0.0076 | Valid Loss: 0.0118 | LR: 0.000002
Epoch [10/10] | Train Loss: 0.0069 | Valid Loss: 0.0119 | LR: 0.000000


In [ ]:
import cv2
import torch
from PIL import Image
import numpy as np

def process_video(model, input_video_path, output_video_path, img_size=224, threshold_cls=0.5):
    model.eval()

    #動画の読み込み設定
    cap = cv2.VideoCapture(input_video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #動画の書き出し設定
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # 画像前処理の定義
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    print("動画処理を開始します...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb_frame)
        input_tensor = transform(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                cls_logits, bbox_pred = model(input_tensor)

            probs = torch.softmax(cls_logits, dim=1).squeeze(0)
            hand_score = probs[1].item()
            pred_bbox = bbox_pred.squeeze(0).cpu().numpy()

        #手が検出された場合赤枠を描画
        if hand_score >= threshold_cls:
            cx, cy, w_rel, h_rel = pred_bbox

            w_px = int(w_rel * width)
            h_px = int(h_rel * height)
            x_px = int((cx * width) - (w_px / 2.0))
            y_px = int((cy * height) - (h_px / 2.0))
            cv2.rectangle(frame, (x_px, y_px), (x_px + w_px, y_px + h_px), (0, 0, 255), 3)
            cv2.putText(frame, f"Hand: {hand_score*100:.1f}%", (x_px, max(20, y_px - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        #処理後のフレームを動画へ書き出し
        out.write(frame)

    cap.release()
    out.release()
    print(f"処理が完了しました: {output_video_path}")

# 実行例
input_path = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4'
output_path = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/output_video.mp4'

process_video(model, input_path, output_path)

動画処理を開始します...
処理が完了しました: /content/drive/MyDrive/kaggle用/画像識別モデル開発/output_video.mp4
